# Encoding dirty categories

When a categorical feature is typed in by hand — a city, a company, a job title — the same
value shows up spelled a dozen different ways: `London`, `london`, `LONDON`, `Lomdon`, `London, UK`.
A one-hot encoder treats every spelling as a brand new, unrelated category. On a stream this is a
double problem: the number of categories keeps growing, and the model never gets to reuse what it
learned about `London` when it later sees `Lomdon`.

The `preprocessing.GapEncoder` tackles this. It represents each string by its character n-gram
counts and factorizes them into a handful of latent topics, so that strings which share n-grams —
which is exactly what misspellings of the same word do — end up with similar encodings. It is the
online counterpart of [skrub's `GapEncoder`](https://skrub-data.org/stable/reference/generated/skrub.GapEncoder.html),
learning one record at a time.

To see when it helps, we'll build a stream where the label depends on the *true* city, but the model
only ever sees a *misspelled* version of it.

## A stream of hand-typed cities

Each customer belongs to a city, and each city has its own churn propensity (the signal we want to
recover). We then corrupt the city name the way a human would: random case, a dropped or swapped
character, sometimes a trailing country code. The label is drawn from the *true* city's propensity,
so a model that could read the clean city name would do well — but our models only see the corrupted
string.

In [1]:
import random

CITIES = {
    "London": 0.15, "Manchester": 0.25, "Birmingham": 0.30,
    "Paris": 0.55, "Marseille": 0.65, "Lyon": 0.60,
    "Berlin": 0.80, "Munich": 0.75, "Hamburg": 0.85,
}


def corrupt(s, rng):
    """Turn a clean city name into something a human might have typed."""
    out = s
    r = rng.random()
    if r < 0.25:
        out = out.lower()
    elif r < 0.4:
        out = out.upper()
    if rng.random() < 0.4 and len(out) > 3:  # drop or swap a character
        i = rng.randrange(1, len(out) - 1)
        if rng.random() < 0.5:
            out = out[:i] + out[i + 1:]
        else:
            out = out[:i] + out[i + 1] + out[i] + out[i + 2:]
    if rng.random() < 0.3:  # trailing region/country, or nothing
        out = out + rng.choice([", UK", ", FR", ", DE", " city", ""])
    return out


def generate(n, seed=42):
    rng = random.Random(seed)
    for _ in range(n):
        city = rng.choice(list(CITIES))
        y = rng.random() < CITIES[city]
        yield city, corrupt(city, rng), y


records = list(generate(4000))
dataset = [({"city": dirty}, y) for _, dirty, y in records]
dataset[:5]

[({'city': 'MACNHESTER'}, True),
 ({'city': 'mancehster, FR'}, False),
 ({'city': 'Hmaburg city'}, True),
 ({'city': 'lyon city'}, True),
 ({'city': 'MNACHESTER'}, False)]

Nine real cities, but a lot more spellings once humans get involved:

In [2]:
spellings = {x["city"] for x, _ in dataset}
print(f"{len(spellings)} distinct spellings for {len(CITIES)} real cities")
print(sorted(s for s in spellings if s.lower().startswith("lo") or "ond" in s.lower())[:10])

689 distinct spellings for 9 real cities
['LODNON', 'LODNON city', 'LODNON, DE', 'LODNON, FR', 'LODON', 'LODON, DE', 'LON', 'LON, UK', 'LONDNO', 'LONDNO city']


## Baseline: one-hot encoding

We evaluate with progressive validation: for each record the model predicts first, then learns.
The metric is ROC AUC, which is well suited to this unbalanced target. The one-hot pipeline learns a
separate weight per spelling, so a spelling it has never seen carries no information.

In [3]:
from river import compose, linear_model, metrics, preprocessing


def evaluate(model, dataset):
    auc = metrics.ROCAUC()
    for x, y in dataset:
        y_pred = model.predict_proba_one(x)
        auc.update(y, y_pred.get(True, 0.0))
        model.learn_one(x, y)
    return auc


one_hot = compose.Pipeline(
    preprocessing.OneHotEncoder(),
    linear_model.LogisticRegression(),
)
evaluate(one_hot, dataset)

ROCAUC: 57.08%

Barely better than a coin flip. With hundreds of one-off spellings, most predictions are made on a
category the model is seeing for the first time.

## GapEncoder

Now we swap the one-hot step for a `GapEncoder` with 10 topics. It reads the text from the `city`
field, turns it into character n-grams, and emits 10 continuous features. Because misspellings of the
same city share n-grams, they map to nearby encodings, and the logistic regression can reuse what it
learned across spellings.

In [4]:
gap = compose.Pipeline(
    preprocessing.GapEncoder(on="city", n_components=10, seed=42),
    linear_model.LogisticRegression(),
)
evaluate(gap, dataset)

ROCAUC: 70.48%

A large jump. To see how much of the signal that recovers, here is an *oracle* that cheats by
one-hot encoding the **clean** city name — the best any model could do given this data:

In [5]:
clean = [({"city": city}, y) for city, _, y in records]

oracle = compose.Pipeline(
    preprocessing.OneHotEncoder(),
    linear_model.LogisticRegression(),
)
evaluate(oracle, clean)

ROCAUC: 71.62%

The `GapEncoder` lands right next to the oracle, even though it only ever saw the misspelled
strings. It recovered almost all of the signal that one-hot encoding threw away.

## Why it works: fuzzy strings share n-grams

Let's train a `GapEncoder` on its own and look at the encodings directly. We compare strings with
cosine similarity: variants of the same city should be close to 1, different cities close to 0.

In [6]:
import math

encoder = preprocessing.GapEncoder(on="city", n_components=10, seed=42)
for x, _ in dataset:
    encoder.learn_one({"city": x["city"]})


def similarity(a, b):
    va = encoder.transform_one({"city": a})
    vb = encoder.transform_one({"city": b})
    dot = sum(va[k] * vb[k] for k in va)
    na = math.sqrt(sum(v * v for v in va.values()))
    nb = math.sqrt(sum(v * v for v in vb.values()))
    return dot / (na * nb + 1e-12)


pairs = [
    ("London", "Lomdon"),
    ("London", "LONDON"),
    ("London", "Paris"),
    ("Paris", "Marseille"),
    ("Berlin", "Munich"),
]
for a, b in pairs:
    print(f"{a:>10} / {b:<10} -> {similarity(a, b):.2f}")

    London / Lomdon     -> 1.00
    London / LONDON     -> 1.00
    London / Paris      -> 0.02
     Paris / Marseille  -> 0.02
    Berlin / Munich     -> 0.02


Misspellings and case changes of the same city sit right on top of each other, while different
cities are essentially orthogonal — including the typo `Lomdon`, which the encoder handles even
though it is just one more never-before-seen spelling.

## Inspecting the topics

Each of the 10 topics tends to specialise in one city. For every topic we can find the training
string that activates it the most, which gives the topics human-readable labels:

In [7]:
unique = sorted({x["city"] for x, _ in dataset})
best = {}
for s in unique:
    h = encoder.transform_one({"city": s})
    topic = max(h, key=h.get)
    if topic not in best or h[topic] > best[topic][1]:
        best[topic] = (s, h[topic])

for topic in sorted(best):
    print(f"topic {topic}: {best[topic][0]!r}")

topic 0: 'MANCHESTER, UK'
topic 1: 'MUNICH, UK'
topic 2: 'BIRMINGHAM city'
topic 3: 'BERLIN city'
topic 4: 'Paris, UK'
topic 5: 'LYON, DE'
topic 6: 'LONDON city'
topic 7: 'MARSIELLE'
topic 8: 'HAMBURG, FR'
topic 9: 'MARSEILLE, FR'


## When to reach for it

The `GapEncoder` earns its place when a categorical feature is **high-cardinality and messy** —
free-typed cities, company names, product labels, job titles, chat handles — where the interesting
structure lives in shared substrings rather than exact matches. It gives you a small, fixed-width,
continuous encoding that generalises across spellings, all learned online. For clean, low-cardinality
categories a plain `OneHotEncoder` is simpler and just as good; for full free text, reach for
`feature_extraction.BagOfWords`/`TFIDF` instead.